In [1]:
import os
import random
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
VALIDATION_FRACTION = 0.10

print("Ground truth:", GROUND_TRUTH)
print("Validation fraction:", VALIDATION_FRACTION)
print("Random seed:", RANDOM_SEED)

Ground truth: ../data/train/train_ground_truth.tsv
Validation fraction: 0.1
Random seed: 42


In [2]:
source1_path = os.path.join(TRAIN_DIR, "train_source1.tsv")

s1_ids = []

for chunk in pd.read_csv(
    source1_path,
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    chunksize=100_000,
    keep_default_na=False
):
    s1_ids.extend(chunk["entity_id"].tolist())

print("Total S1 IDs:", len(s1_ids))
print("Unique S1 IDs:", len(set(s1_ids)))

Total S1 IDs: 2206821
Unique S1 IDs: 2206821


In [3]:
rng = random.Random(RANDOM_SEED)

shuffled_ids = s1_ids.copy()
rng.shuffle(shuffled_ids)

validation_size = int(len(shuffled_ids) * VALIDATION_FRACTION)

validation_s1_ids = set(shuffled_ids[:validation_size])
train_s1_ids = set(shuffled_ids[validation_size:])

print("Total S1:", len(shuffled_ids))
print("Training S1:", len(train_s1_ids))
print("Validation S1:", len(validation_s1_ids))

print(
    "Validation percentage:",
    len(validation_s1_ids) / len(shuffled_ids) * 100
)

Total S1: 2206821
Training S1: 1986139
Validation S1: 220682
Validation percentage: 9.99999546859487


In [4]:
validation_ids_path = os.path.join(
    OUTPUT_DIR,
    "validation_s1_ids.txt"
)

train_ids_path = os.path.join(
    OUTPUT_DIR,
    "train_s1_ids.txt"
)

with open(validation_ids_path, "w", encoding="utf-8") as f:
    for entity_id in sorted(validation_s1_ids):
        f.write(entity_id + "\n")

with open(train_ids_path, "w", encoding="utf-8") as f:
    for entity_id in sorted(train_s1_ids):
        f.write(entity_id + "\n")

print("Saved:")
print(validation_ids_path)
print(train_ids_path)

Saved:
../output/validation_s1_ids.txt
../output/train_s1_ids.txt


In [5]:
def f05_single_entity(true_ids, predicted_ids):
    """
    Compute F0.5 for one Source 1 entity.
    """

    true_set = set(true_ids)
    predicted_set = set(predicted_ids)

    # True singleton
    if len(true_set) == 0:
        return 1.0 if len(predicted_set) == 0 else 0.0

    # True matches exist, but prediction is empty
    if len(predicted_set) == 0:
        return 0.0

    true_positives = len(true_set & predicted_set)

    precision = true_positives / len(predicted_set)
    recall = true_positives / len(true_set)

    denominator = 0.25 * precision + recall

    if denominator == 0:
        return 0.0

    return (
        1.25 * precision * recall
        / denominator
    )

In [6]:
def macro_f05(ground_truth, predictions):
    """
    ground_truth:
        dict {S1_ID: set(S2/S3 IDs)}

    predictions:
        dict {S1_ID: set(S2/S3 IDs)}
    """

    scores = []

    for s1_id, true_ids in ground_truth.items():
        predicted_ids = predictions.get(s1_id, set())

        score = f05_single_entity(
            true_ids,
            predicted_ids
        )

        scores.append(score)

    if not scores:
        return 0.0

    return float(np.mean(scores))

In [7]:
validation_ground_truth = {}

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for _, row in chunk.iterrows():

        s1_id = row["source1_entity_id"]

        if s1_id not in validation_s1_ids:
            continue

        value = row["matched_entity_ids"].strip()

        if value == "":
            matches = set()
        else:
            matches = set(
                x.strip()
                for x in value.split(",")
                if x.strip()
            )

        validation_ground_truth[s1_id] = matches

print(
    "Validation ground-truth entities:",
    len(validation_ground_truth)
)

Validation ground-truth entities: 220682


In [8]:
missing_validation_labels = (
    validation_s1_ids - validation_ground_truth.keys()
)

extra_validation_labels = (
    validation_ground_truth.keys() - validation_s1_ids
)

print("Validation S1 IDs:", len(validation_s1_ids))
print("Ground-truth entries:", len(validation_ground_truth))

print("Missing labels:", len(missing_validation_labels))
print("Extra labels:", len(extra_validation_labels))

Validation S1 IDs: 220682
Ground-truth entries: 220682
Missing labels: 0
Extra labels: 0


In [9]:
empty_predictions = {
    s1_id: set()
    for s1_id in validation_s1_ids
}

baseline_f05 = macro_f05(
    validation_ground_truth,
    empty_predictions
)

print("Empty-prediction macro F0.5:", baseline_f05)

Empty-prediction macro F0.5: 0.05642961365222356


In [10]:
validation_match_distribution = Counter(
    len(matches)
    for matches in validation_ground_truth.values()
)

validation_distribution = pd.DataFrame(
    sorted(
        validation_match_distribution.items(),
        key=lambda x: x[0]
    ),
    columns=[
        "match_count",
        "s1_entities"
    ]
)

validation_distribution["percentage"] = (
    validation_distribution["s1_entities"]
    / len(validation_ground_truth)
    * 100
)

validation_distribution

,match_count,s1_entities,percentage
0,0,12453,5.642961
1,1,11935,5.408234
2,2,37421,16.956979
3,3,52941,23.989723
4,4,48524,21.988200
5,5,31944,14.475127
6,6,16577,7.511714
7,7,6489,2.940430
8,8,1903,0.862327
9,9,436,0.197569


In [11]:
validation_zero_matches = validation_match_distribution.get(0, 0)

print(
    "Validation singleton rate:",
    validation_zero_matches
    / len(validation_ground_truth)
    * 100
)

Validation singleton rate: 5.642961365222356


In [12]:
target_to_s1 = defaultdict(set)

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=100_000,
    keep_default_na=False
):
    for _, row in chunk.iterrows():

        s1_id = row["source1_entity_id"]
        value = row["matched_entity_ids"].strip()

        if value == "":
            continue

        for target_id in value.split(","):
            target_id = target_id.strip()

            if target_id:
                target_to_s1[target_id].add(s1_id)

multi_owner_targets = {
    target_id: owners
    for target_id, owners in target_to_s1.items()
    if len(owners) > 1
}

print(
    "Unique S2/S3 IDs appearing in ground truth:",
    len(target_to_s1)
)

print(
    "S2/S3 IDs linked to multiple S1 entities:",
    len(multi_owner_targets)
)

Unique S2/S3 IDs appearing in ground truth: 7638365
S2/S3 IDs linked to multiple S1 entities: 0
